# Notebook 03 — Identification
**Ahead of the Curve: EV Smart Charging and GB Evening Demand Shape**

---

> **This notebook only runs if Notebook 02 passed the hard gate.**  
> The first cell checks `gate_state.json`. If the gate failed, execution stops here.

---

## What this notebook does

1. **Gate enforcement** — refuses to run if Notebook 02 gate failed
2. **DiD specification** — estimates β (spread ~ compliant_stock + controls)
3. **Variance regression** — estimates the change in evening price variance
4. **Pre-2021 falsification** — checks the effect doesn't appear before treatment
5. **Placebo null** — verifies the pipeline doesn't generate spurious signal
6. **Parallel trends** — visual and statistical test of the DiD assumption
7. **Interconnector control** — robustness check for continental price effects
8. **Seasonality check** — winter effect should exceed summer effect
9. **Sensitivity** — compliance rates 30%, 50%, 70%
10. **Output** — β estimates and variance parameters fed to Notebook 04 (dispatch)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})
COLOUR_SPREAD  = '#2C5F8A'
COLOUR_STOCK   = '#C65B2A'
COLOUR_NEUTRAL = '#888888'

DATA_DIR = Path('../data')
FIG_DIR  = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)

MANDATE_DATE    = '2022-06-01'
COMPLIANCE_RATE = 0.50

print('Notebook 03 — Identification')

## 0. Gate enforcement

Hard stop if Notebook 02 did not pass.

In [ ]:
gate_path = DATA_DIR / 'gate_state.json'

if not gate_path.exists():
    raise RuntimeError(
        'gate_state.json not found.\n'
        'Run Notebook 02 (02_raw_inspection.ipynb) first.'
    )

with open(gate_path) as f:
    gate = json.load(f)

print(f'Gate state loaded (run: {gate["run_date"]})')
print(f'  ρ (partial, post-mandate): {gate["rho_partial"]:+.3f}')
print(f'  p (partial):               {gate["p_partial"]:.3f}')
print()

if not gate['gate_passed']:
    raise RuntimeError(
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        '  GATE FAILED — NOTEBOOK 03 WILL NOT RUN  \n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'  Signal below detection threshold at current EV penetration.\n'
        f'  See gate_state.json for minimum detectable stock estimate.\n'
        f'  Project stops here. No exceptions.'
    )

print('✓ Gate passed. Proceeding with identification.')

## 1. Load panel data

In [ ]:
# Load daily panel from Notebook 02
panel   = pd.read_parquet(DATA_DIR / 'panel_daily.parquet')
monthly = pd.read_parquet(DATA_DIR / 'panel_monthly.parquet')

# Load demand and interconnector
def load_if_exists(fname):
    p = DATA_DIR / fname
    return pd.read_parquet(p) if p.exists() else None

demand_raw      = load_if_exists('demand_raw.parquet')
interconnector  = load_if_exists('interconnector_raw.parquet')
storage_raw     = load_if_exists('storage_raw.parquet')

# Merge additional controls into panel
if demand_raw is not None:
    demand_raw.index = pd.DatetimeIndex(demand_raw.index).tz_localize(None)
    demand_col = demand_raw.columns[0]
    demand_daily = demand_raw[[demand_col]].rename(columns={demand_col: 'demand_mw'}).resample('D').mean()
    panel = panel.join(demand_daily, how='left')
    print(f'Demand merged: {demand_daily.columns.tolist()}')
else:
    panel['demand_mw'] = np.nan
    print('⚠ Demand data not available — demand control will be excluded')

if interconnector is not None:
    interconnector.index = pd.DatetimeIndex(interconnector.index).tz_localize(None)
    ic_daily = interconnector.resample('D').mean()
    panel = panel.join(ic_daily, how='left')
    print(f'Interconnector merged')
else:
    panel['net_interconnector_mw'] = np.nan
    print('⚠ Interconnector data not available — will be excluded from DiD')

if storage_raw is not None:
    storage_raw.index = pd.DatetimeIndex(storage_raw.index).tz_localize(None)
    storage_daily = storage_raw.resample('D').mean()
    panel = panel.join(storage_daily, how='left')
    print(f'Storage merged')
else:
    panel['battery_mw'] = np.nan
    print('⚠ Storage data not available — will be excluded from DiD')

# Season and gas price dummies (gas price requires external source — use month FE as proxy)
panel['season'] = pd.cut(
    panel.index.month,
    bins=[0, 3, 6, 9, 12],
    labels=['Q1_winter', 'Q2_spring', 'Q3_summer', 'Q4_autumn']
)
panel['post_mandate'] = (panel.index >= pd.Timestamp(MANDATE_DATE)).astype(int)

print(f'\nPanel: {len(panel):,} days  |  cols: {list(panel.columns)}')

## 2. DiD specification

```
spread(t) = α
          + β  × compliant_stock(t)
          + γ  × wind_forecast_error(t)
          + δ  × demand(t)                [if available]
          + ζ  × net_interconnector_flow(t) [if available]
          + season_fe
          + u(t)
```

β is the estimate of interest: £/MWh per additional compliant chargepoint.  
Converted to £/MWh per GW of shiftable load for interpretability.

Standard errors: Newey-West HAC (accounts for serial correlation in daily price data).

In [ ]:
def build_formula(panel: pd.DataFrame) -> str:
    """
    Build regression formula based on which controls are available.
    """
    base = 'arithmetic_spread ~ compliant_stock + wind_forecast_error'
    
    if panel['demand_mw'].notna().sum() > 100:
        base += ' + demand_mw'
    if panel['net_interconnector_mw'].notna().sum() > 100:
        base += ' + net_interconnector_mw'
    if panel['battery_mw'].notna().sum() > 100:
        base += ' + battery_mw'
    
    # Season fixed effects
    base += ' + C(season)'
    # Year fixed effects (absorbs gas price trend)
    base += ' + C(year)'
    
    return base


# Full sample DiD
clean = panel.dropna(subset=['arithmetic_spread', 'compliant_stock', 'wind_forecast_error'])
formula = build_formula(clean)
print('DiD formula:')
print(f'  {formula}')
print()

model_full = smf.ols(formula, data=clean).fit(
    cov_type='HAC', cov_kwds={'maxlags': 10}  # Newey-West, 10 lags ≈ 5 days
)
print(model_full.summary2())

In [ ]:
# Extract β and convert to interpretable units
beta     = model_full.params['compliant_stock']
beta_se  = model_full.bse['compliant_stock']
beta_p   = model_full.pvalues['compliant_stock']
beta_ci  = model_full.conf_int().loc['compliant_stock']

# Convert: £/MWh per chargepoint → £/MWh per GW shiftable
chargepoints_per_gw = 1e6 / (7.0 * 0.30)  # ≈ 476,190 chargepoints per GW
beta_per_gw    = beta    * chargepoints_per_gw
beta_se_per_gw = beta_se * chargepoints_per_gw
beta_ci_gw     = beta_ci * chargepoints_per_gw

print('=== DiD RESULT — β (compliant_stock) ===')
print()
print(f'  β (£/MWh per chargepoint):    {beta:+.6f}  SE={beta_se:.6f}  p={beta_p:.4f}')
print(f'  β (£/MWh per GW shiftable):   {beta_per_gw:+.2f}  SE={beta_se_per_gw:.2f}')
print(f'  95% CI (per GW):              [{beta_ci_gw.iloc[0]:+.2f}, {beta_ci_gw.iloc[1]:+.2f}]')
print()
print(f'  Interpretation: each additional GW of shiftable evening EV load')
print(f'  is associated with a {beta_per_gw:+.1f} £/MWh change in the evening spread.')
print(f'  Expected sign: negative (smart charging flattens the ramp).')

## 3. Variance regression

The mean spread shift (β) is only half the story.  
Spike suppression — reduction in the right tail — often dominates NPV.

Method: GLS-style regression of squared evening price on compliant stock.  
Also: logistic regression of spike indicator on compliant stock.

In [ ]:
import statsmodels.formula.api as smf

# ── Variance of evening prices ~ compliant stock ───────────────────────────────
# Use monthly panels to reduce noise
monthly_clean = monthly.dropna(subset=['mean_spread', 'compliant_stock'])

# We don't have raw monthly variance from the panel; compute it
monthly_var = panel.groupby(panel.index.to_period('M')).agg(
    evening_var=('arithmetic_spread', 'var'),
    spike_freq=('evening_spike_freq', 'mean'),
    compliant_stock=('compliant_stock', 'mean'),
    wind_forecast_error=('wind_forecast_error', 'mean'),
    year=('year', 'first'),
    season=('season', 'first'),
).dropna(subset=['evening_var', 'compliant_stock'])

# Variance model
var_formula = 'evening_var ~ compliant_stock + wind_forecast_error + C(season) + C(year)'
model_var = smf.ols(var_formula, data=monthly_var).fit(
    cov_type='HAC', cov_kwds={'maxlags': 3}
)

gamma_var = model_var.params['compliant_stock']
gamma_p   = model_var.pvalues['compliant_stock']

print('=== VARIANCE REGRESSION ===')
print()
print(f'  γ (variance per chargepoint): {gamma_var:+.4f}  p={gamma_p:.4f}')
print(f'  γ (variance per GW):          {gamma_var * chargepoints_per_gw:+.1f}')
print()

# Spike frequency model
spike_formula = 'spike_freq ~ compliant_stock + wind_forecast_error + C(season) + C(year)'
model_spike = smf.ols(spike_formula, data=monthly_var).fit(
    cov_type='HAC', cov_kwds={'maxlags': 3}
)

delta_spike = model_spike.params['compliant_stock']
delta_p     = model_spike.pvalues['compliant_stock']

print(f'=== SPIKE FREQUENCY REGRESSION ===')
print()
print(f'  δ (spike freq per chargepoint): {delta_spike:+.8f}  p={delta_p:.4f}')
print(f'  δ (spike freq per GW):          {delta_spike * chargepoints_per_gw:+.4f}  (pp change)')

## 4. Falsification — pre-2021 subsample

Before GB battery storage exceeded ~500 MW, the treatment (smart charging) was also negligible.  
If β appears in the pre-2021 subsample → identification is compromised by a confound.  

Expected result: β not significant pre-2021.

In [ ]:
pre2021 = clean.loc[clean.index < pd.Timestamp('2021-01-01')].copy()

if len(pre2021) < 100:
    print('⚠ Insufficient pre-2021 data for falsification test.')
else:
    # In pre-2021 subsample, compliant_stock should be near-zero
    # (mandate not yet in effect; almost no smart chargers)
    # Use a simpler formula without year FE (not enough years)
    formula_pre = 'arithmetic_spread ~ compliant_stock + wind_forecast_error + C(season)'
    model_pre = smf.ols(formula_pre, data=pre2021).fit(
        cov_type='HAC', cov_kwds={'maxlags': 10}
    )
    
    beta_pre   = model_pre.params['compliant_stock']
    beta_pre_p = model_pre.pvalues['compliant_stock']
    
    print('=== FALSIFICATION — PRE-2021 SUBSAMPLE ===')
    print()
    print(f'  β (pre-2021):  {beta_pre:+.6f}  p={beta_pre_p:.4f}')
    print(f'  β (full):      {beta:+.6f}  p={beta_p:.4f}')
    print()
    
    if beta_pre_p < 0.05 and abs(beta_pre) > abs(beta) * 0.5:
        print('  ⚠  WARNING: Pre-treatment effect is significant and large.')
        print('     Identification may be compromised by a confound.')
        print('     FALSIFICATION CONDITION 2 TRIGGERED.')
        FALSIFICATION_2_FAILED = True
    else:
        print('  ✓ Pre-2021 effect not significant (or small).')
        print('    Falsification condition 2 passes.')
        FALSIFICATION_2_FAILED = False

## 5. Placebo null

Set `compliant_stock = 0` throughout and rerun the regression.  
β should not be significant — if it is, the pipeline is generating spurious signal.

In [ ]:
placebo_data = clean.copy()
placebo_data['compliant_stock'] = 0.0

model_placebo = smf.ols(formula, data=placebo_data).fit(
    cov_type='HAC', cov_kwds={'maxlags': 10}
)

# With stock=0 everywhere, the coefficient is undefined/degenerate — just check R²
# Better placebo: randomly shuffle compliant_stock across dates
np.random.seed(42)
shuffled_data = clean.copy()
shuffled_data['compliant_stock'] = np.random.permutation(clean['compliant_stock'].values)

model_shuffled = smf.ols(formula, data=shuffled_data).fit(
    cov_type='HAC', cov_kwds={'maxlags': 10}
)

beta_placebo   = model_shuffled.params['compliant_stock']
beta_placebo_p = model_shuffled.pvalues['compliant_stock']

print('=== PLACEBO NULL (shuffled treatment) ===')
print()
print(f'  β (shuffled): {beta_placebo:+.6f}  p={beta_placebo_p:.4f}')
print(f'  β (actual):   {beta:+.6f}  p={beta_p:.4f}')
print()

if beta_placebo_p < 0.05:
    print('  ⚠  WARNING: Placebo significant. Pipeline may generate spurious signal.')
    print('     FALSIFICATION CONDITION 3 TRIGGERED.')
    FALSIFICATION_3_FAILED = True
else:
    print('  ✓ Placebo not significant. Falsification condition 3 passes.')
    FALSIFICATION_3_FAILED = False

## 6. Parallel trends

Tests whether the pre-mandate trend in spread is parallel across high/low EV adoption areas.  

Since this is a national (GB) panel without regional variation, we use a time-series proxy:  
**Pre-mandate:** compare winters (high EV demand) vs summers (low EV demand).  
The spread should trend similarly across seasons before treatment.  
Post-mandate: winters should show a larger effect if the signal is real.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Parallel Trends and Falsification', fontsize=12, fontweight='bold')

# ── Panel 1: Pre vs post spread by season ─────────────────────────────────────
ax = axes[0]
quarterly = panel.resample('QE').agg(
    mean_spread=('arithmetic_spread', 'mean'),
    compliant_stock=('compliant_stock', 'mean'),
    season=('season', 'first'),
)

for season, colour in [('Q1_winter', COLOUR_SPREAD), ('Q3_summer', COLOUR_NEUTRAL)]:
    s = quarterly[quarterly['season'] == season]
    ax.plot(s.index, s['mean_spread'], label=season, color=colour, lw=1.5, marker='o', ms=4)

ax.axvline(pd.Timestamp(MANDATE_DATE), color=COLOUR_STOCK, lw=1.5, ls='--', label='Mandate')
ax.set_title('Spread by Season (Parallel Trends Proxy)')
ax.set_ylabel('Mean spread (£/MWh)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))

# ── Panel 2: Event study — annual β estimates ──────────────────────────────────
ax = axes[1]

# Run year-by-year β to show event study pattern
years = sorted(clean['year'].unique())
betas, cis, yr_labels = [], [], []

for yr in years:
    yr_data = clean[clean['year'] == yr].copy()
    if len(yr_data) < 50:
        continue
    try:
        yr_formula = 'arithmetic_spread ~ compliant_stock + wind_forecast_error + C(season)'
        m = smf.ols(yr_formula, data=yr_data).fit(cov_type='HC3')
        b = m.params['compliant_stock']
        ci = m.conf_int().loc['compliant_stock'].values
        betas.append(b * chargepoints_per_gw)
        cis.append(ci * chargepoints_per_gw)
        yr_labels.append(yr)
    except Exception:
        pass

if betas:
    cis = np.array(cis)
    ax.axhline(0, color='black', lw=0.8, ls=':')
    ax.axvline(
        yr_labels.index(min([y for y in yr_labels if y >= 2022], default=yr_labels[-1])),
        color=COLOUR_STOCK, lw=1.5, ls='--', label='Mandate'
    )
    ax.errorbar(
        range(len(yr_labels)), betas,
        yerr=[np.array(betas) - cis[:, 0], cis[:, 1] - np.array(betas)],
        fmt='o', color=COLOUR_SPREAD, capsize=4, lw=1.5
    )
    ax.set_xticks(range(len(yr_labels)))
    ax.set_xticklabels(yr_labels, rotation=45)
    ax.set_title('Event Study: Annual β Estimates (£/MWh per GW)')
    ax.set_ylabel('β (£/MWh per GW shiftable load)')
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / 'nb03_identification.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIG_DIR}/nb03_identification.png')

## 7. Sensitivity — compliance rate

Pre-registered sensitivity range: 30%, 50% (central), 70%.  
β is reparameterised for each compliance rate.

In [ ]:
# Load chargepoints to rebuild compliant stock at different compliance rates
chargepoints = pd.read_parquet(DATA_DIR / 'chargepoints.parquet')

MANDATE_TS = pd.Timestamp(MANDATE_DATE)

def rebuild_compliant_stock(cp_df: pd.DataFrame, compliance_rate: float) -> pd.Series:
    """
    Rebuild compliant_stock at a given compliance rate.
    Uses total_home_chargepoints_cumulative column.
    """
    cp = cp_df.copy()
    home_col = 'home_chargepoints_cumulative'
    if home_col not in cp.columns:
        home_col = cp.columns[0]
    
    baseline = cp.loc[cp.index < MANDATE_TS, home_col].iloc[-1] if len(cp.loc[cp.index < MANDATE_TS]) > 0 else 0
    incremental = (cp[home_col] - baseline).clip(lower=0)
    incremental[cp.index < MANDATE_TS] = 0
    return incremental * compliance_rate


sensitivity_results = {}
print('=== SENSITIVITY — COMPLIANCE RATE ===')
print()
print(f'  {"Compliance":>12}  {"β (per GW)":>12}  {"SE":>8}  {"p":>8}')
print('  ' + '-' * 45)

for cr in [0.30, 0.50, 0.70]:
    try:
        stock_series = rebuild_compliant_stock(chargepoints, cr)
        stock_daily  = stock_series.resample('D').ffill()
        stock_daily.index = pd.DatetimeIndex(stock_daily.index).tz_localize(None)
        
        sens_panel = clean.copy()
        sens_panel['compliant_stock'] = stock_daily.reindex(sens_panel.index, method='ffill').fillna(0)
        
        m = smf.ols(formula, data=sens_panel).fit(cov_type='HAC', cov_kwds={'maxlags': 10})
        b  = m.params['compliant_stock']
        se = m.bse['compliant_stock']
        p  = m.pvalues['compliant_stock']
        
        sensitivity_results[cr] = {
            'beta': b, 'beta_per_gw': b * chargepoints_per_gw,
            'se': se, 'p': p
        }
        print(f'  {cr:>12.0%}  {b * chargepoints_per_gw:>+12.2f}  {se * chargepoints_per_gw:>8.2f}  {p:>8.4f}')
    except Exception as e:
        print(f'  {cr:.0%}  ERROR: {e}')

## 8. Seasonality check

Smart charging should suppress the evening ramp more in winter (early dark, high demand)  
than summer (late sunset, lower demand, longer EV charging window).

Run β separately for winter (Nov–Feb) and summer (May–Aug).  
Expected: |β_winter| > |β_summer|.

In [ ]:
winter = clean[clean['month'].isin([11, 12, 1, 2])]
summer = clean[clean['month'].isin([5, 6, 7, 8])]

season_formula = 'arithmetic_spread ~ compliant_stock + wind_forecast_error + C(year)'

results_season = {}
for label, subset in [('Winter (Nov-Feb)', winter), ('Summer (May-Aug)', summer)]:
    if len(subset) < 100:
        continue
    m = smf.ols(season_formula, data=subset).fit(cov_type='HAC', cov_kwds={'maxlags': 10})
    b  = m.params['compliant_stock']
    p  = m.pvalues['compliant_stock']
    results_season[label] = {'beta': b, 'beta_per_gw': b * chargepoints_per_gw, 'p': p}

print('=== SEASONALITY CHECK ===')
print()
for label, res in results_season.items():
    print(f'  {label:<22}  β={res["beta_per_gw"]:+.2f} £/MWh per GW   p={res["p"]:.4f}')

print()
if len(results_season) == 2:
    betas = [r['beta_per_gw'] for r in results_season.values()]
    if abs(betas[0]) > abs(betas[1]):
        print('  ✓ Winter effect > Summer effect. Consistent with mechanism.')
    else:
        print('  ⚠ Summer effect ≥ Winter effect. Unexpected. Review.')

## 9. Falsification summary

In [ ]:
print('=== FALSIFICATION CONDITIONS SUMMARY ===')
print()

conditions = [
    ('Condition 1', 'Hard gate (NB02)',         True,  'Passed — we are here'),
    ('Condition 2', 'Pre-2021 effect',           not FALSIFICATION_2_FAILED, 'Pre-treatment β not significant'),
    ('Condition 3', 'Placebo null',              not FALSIFICATION_3_FAILED, 'Shuffled stock β not significant'),
    ('Condition 4', 'β significant post-2021',  beta_p < 0.10, f'p={beta_p:.4f}'),
]

all_pass = True
for cond, label, passes, detail in conditions:
    icon = '✓' if passes else '✗'
    if not passes:
        all_pass = False
    print(f'  {icon}  {cond}: {label:<30} {detail}')

print()
if all_pass:
    print('  All pre-registered falsification conditions pass.')
    print('  Proceed to Notebook 04 (dispatch model).')
else:
    print('  ⚠ One or more falsification conditions triggered.')
    print('    Review carefully before proceeding to Notebook 04.')

## 10. Save identification results for Notebook 04

In [ ]:
identification_output = {
    # Mean spread coefficient
    'beta_per_chargepoint':    float(beta),
    'beta_per_gw':             float(beta_per_gw),
    'beta_se_per_gw':          float(beta_se_per_gw),
    'beta_p':                  float(beta_p),
    'beta_ci_lower':           float(beta_ci_gw.iloc[0]),
    'beta_ci_upper':           float(beta_ci_gw.iloc[1]),
    
    # Variance / spike
    'gamma_var_per_chargepoint': float(gamma_var),
    'gamma_var_per_gw':          float(gamma_var * chargepoints_per_gw),
    'delta_spike_per_chargepoint': float(delta_spike),
    'delta_spike_per_gw':          float(delta_spike * chargepoints_per_gw),
    
    # Sensitivity (compliance rates)
    'sensitivity_30pct_beta_per_gw': float(sensitivity_results.get(0.30, {}).get('beta_per_gw', np.nan)),
    'sensitivity_50pct_beta_per_gw': float(sensitivity_results.get(0.50, {}).get('beta_per_gw', np.nan)),
    'sensitivity_70pct_beta_per_gw': float(sensitivity_results.get(0.70, {}).get('beta_per_gw', np.nan)),
    
    # Model info
    'n_obs':       int(len(clean)),
    'r_squared':   float(model_full.rsquared),
    'formula':     formula,
    'run_date':    pd.Timestamp.today().strftime('%Y-%m-%d'),
    
    # Falsification
    'falsification_2_failed': FALSIFICATION_2_FAILED,
    'falsification_3_failed': FALSIFICATION_3_FAILED,
    'all_falsification_pass': all_pass,
    
    # Season checks
    'season_check': {k: {kk: float(vv) for kk, vv in v.items()} for k, v in results_season.items()},
}

out_path = DATA_DIR / 'identification_results.json'
with open(out_path, 'w') as f:
    json.dump(identification_output, f, indent=2)

print(f'Identification results saved → {out_path}')
print()
print('Summary:')
print(f'  β  = {beta_per_gw:+.2f} £/MWh per GW shiftable  (p={beta_p:.4f})')
print(f'  γ  = {gamma_var * chargepoints_per_gw:+.1f} var units per GW  (p={gamma_p:.4f})')
print(f'  δ  = {delta_spike * chargepoints_per_gw:+.4f} spike pp per GW  (p={delta_p:.4f})')
print(f'  R² = {model_full.rsquared:.3f}  N = {len(clean):,} days')